# 7.06 Knowledge · Reference 도구 — Wikipedia · arXiv · PubMed · Google Scholar

에이전트가 **참조 가능한 백과사전·학술 DB**에 직접 질의하도록 붙여주는 도구 4종. 검색 엔진(`07_01`) 과 달리 **구조화된 공식 소스**를 조회하므로 RAG 의 근거 문서로 쓰기 좋다. 네 도구 모두 **키 없이 공개 API** 로 동작 (rate limit 존재).

- `WikipediaQueryRun` — Wikimedia API
- `ArxivQueryRun` — arXiv (물리·수학·CS)
- `PubmedQueryRun` — NCBI E-utilities (생의학)
- Google Scholar — `scholarly` (비공식 래퍼)

## 학습 목표

- 네 도구의 **반환 문자열 포맷** 과 wrapper 파라미터를 비교
- `WikipediaAPIWrapper(lang=...)` 로 다국어 전환, `top_k_results` · `doc_content_chars_max` 로 길이 제어
- 하나의 에이전트에 **여러 지식 도구**를 묶어 라우팅
- 공식 API 도구 (`WikipediaQueryRun` 등) 와 비공식 스크래퍼 (`scholarly`) 의 **안정성·ToS 차이**

## 언제 쓰나

- 개념 정의·배경 지식: Wikipedia
- CS · 수학 · 물리 **최신 연구**: arXiv
- **의학·생명과학** 근거 자료: PubMed (MEDLINE)
- 인용 네트워크·저자 프로필: Google Scholar (주의: ToS 위반 소지, CAPTCHA)

## 7.06.1 환경 설정

Wikipedia · arXiv · PubMed는 키 없이 동작. 외부 패키지 `wikipedia`, `arxiv`, `xmltodict` 만 필요. Google Scholar는 `scholarly` 가 별도 필요(선택).

In [ ]:
# !pip install -U langchain langchain-community wikipedia arxiv xmltodict

import os
from dotenv import load_dotenv
load_dotenv(override=True)

# 이 노트북은 모든 외부 도구가 키 없이 동작 — 외부 라이브러리만 확인
import wikipedia  # noqa: F401
import arxiv      # noqa: F401
import xmltodict  # noqa: F401  # PubMed XML 응답 파서

print("OPENAI_API_KEY 존재:", bool(os.environ.get("OPENAI_API_KEY")))  # 7.06.3 에이전트 섹션에만 사용

## 7.06.2 기본 사용 — 도구 단독 `.invoke(...)`

네 도구 모두 **`QueryRun` 계열** 은 `str -> str` 반환. `APIWrapper` 의 하이퍼파라미터로 반환 길이를 제어한다.

In [ ]:
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.tools.pubmed.tool import PubmedQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(lang="en", top_k_results=1, doc_content_chars_max=400),
)
print("--- Wikipedia ---")
print(wiki_tool.invoke("LangChain"))

arxiv_tool = ArxivQueryRun(
    api_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=400),
)
print("\n--- arXiv ---")
print(arxiv_tool.invoke("retrieval augmented generation"))

pubmed_tool = PubmedQueryRun()  # 기본 wrapper 자동 생성
print("\n--- PubMed ---")
print(pubmed_tool.invoke("gut microbiome depression")[:600])

## 7.06.3 `create_agent` 에 연결 — 지식 도구 라우팅

세 도구를 한 에이전트에 물려주면 **질문 유형별로 모델이 도구를 고른다**. `description` 을 선명하게 쓰는 것이 라우팅 품질의 핵심.

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain.agents import create_agent

    # description을 명시적으로 덮어써 라우팅 품질 향상
    wiki_tool.description = "개념 정의·백과사전적 사실을 영어 Wikipedia에서 찾는다."
    arxiv_tool.description = "arXiv에서 CS·수학·물리 논문 초록을 찾는다."
    pubmed_tool.description = "PubMed(MEDLINE)에서 의학·생명과학 논문을 찾는다."

    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=[wiki_tool, arxiv_tool, pubmed_tool],
        system_prompt=(
            "질문의 성격에 가장 맞는 지식 도구 하나를 먼저 호출한 뒤, "
            "반환된 내용을 근거로 2~3문장 답한다. 출처 제목을 한 줄 인용."
        ),
    )
    res = agent.invoke({"messages": [{"role": "user", "content": "CRISPR 유전자 편집의 최신 개선 방향을 알려줘"}]})
    print(res["messages"][-1].content[:600])
else:
    print("OPENAI_API_KEY 없음 → 에이전트 예제 스킵. 도구는 7.06.2에서 이미 동작 확인됨.")

## 7.06.4 주요 옵션 · 비교 표

| 도구 | Wrapper 주요 필드 | 반환 단위 | Rate limit | 인증 |
|------|------------------|----------|-----------|------|
| `WikipediaQueryRun` | `lang`, `top_k_results`, `doc_content_chars_max`, `load_all_available_meta` | 페이지 요약/본문 | 관대 (Wikimedia fair use) | 불필요 |
| `ArxivQueryRun` | `top_k_results`, `load_max_docs`, `doc_content_chars_max`, `load_all_available_meta` | 논문 메타 + 초록 | 공손 간격 권장 (≥3초) | 불필요 |
| `PubmedQueryRun` | `top_k_results`, `doc_content_chars_max`, `email`, `api_key` | 논문 메타 + 초록 | 익명 3 req/s, 키 등록 시 10 req/s | 이메일 등록 권장 |
| Google Scholar (`scholarly`) | `search_pubs`, `fill`, … | 인용·저자 프로필 | 비공식 — **CAPTCHA 가능** | ToS 위반 리스크 |

### 다국어 Wikipedia

`lang` 은 ISO 639-1 (`ko`, `ja`, `zh`, `de`, …). 한국어 페이지는 영어 대비 요약 길이가 짧은 항목이 많다.

In [ ]:
wiki_ko = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(lang="ko", top_k_results=1, doc_content_chars_max=500),
)
print(wiki_ko.invoke("트랜스포머 신경망"))

## 7.06.5 (선택) Google Scholar — `scholarly` 래핑

LangChain community 에는 공식 Google Scholar 도구가 없다. 널리 쓰이는 `scholarly` 패키지를 `@tool` 로 감싸 에이전트에 주입한다. **주의**:
- Google Scholar 는 공식 API 가 없어 **ToS 위반 소지**, CAPTCHA 발생 시 IP 차단 가능
- 프로덕션에서는 Semantic Scholar, OpenAlex 같은 공식 API 사용 권장

아래 셀은 `scholarly` 가 설치돼 있고 네트워크가 허용될 때만 실행한다 (기본 스킵).

In [ ]:
ENABLE_SCHOLAR = False  # 활성화하려면 True + `uv pip install scholarly` 필요

if ENABLE_SCHOLAR:
    from langchain.tools import tool
    from scholarly import scholarly

    @tool
    def scholar_search(query: str, k: int = 3) -> str:
        """Google Scholar 에서 상위 k편 논문의 제목·저자·연도·인용수를 반환."""
        hits = []
        it = scholarly.search_pubs(query)
        for _ in range(k):
            try:
                pub = next(it)
            except StopIteration:
                break
            bib = pub.get("bib", {})
            hits.append(
                f"- {bib.get('title')} ({bib.get('pub_year')}) — cited_by={pub.get('num_citations')}"
            )
        return "\n".join(hits) or "검색 결과 없음"

    print(scholar_search.invoke({"query": "retrieval augmented generation", "k": 3}))
else:
    print("Google Scholar 예제 스킵 — ENABLE_SCHOLAR=True 로 바꾸고 scholarly 설치 필요.")

## 7.06.6 (선택) HITL 결합 — 대량 질의 승인

Wikipedia/arXiv/PubMed 단일 호출은 보통 HITL 을 걸 필요가 없다. 그러나 에이전트가 **짧은 간격에 수십 건** 을 쏟아내려 할 때는 rate limit 위반을 막기 위해 사람 승인을 한 번 끼울 수 있다.

```python
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[wiki_tool, arxiv_tool, pubmed_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"arxiv": {"allowed_decisions": ["approve", "reject"]}},
        ),
    ],
)
```

실제 HITL resume 흐름은 `02_langchain/07_hitl_and_runtime` 노트북 참고.

## 체크리스트

- [ ] `WikipediaAPIWrapper(lang="ko")` 로 한국어 문서 질의 가능 확인
- [ ] `ArxivAPIWrapper(load_all_available_meta=True)` 이면 DOI·저자 ORCID 등 메타가 풍부해진다 (응답 파싱 비용 ↑)
- [ ] PubMed 는 **공손한 요청 간격** 을 `email`·`api_key` 등록으로 확보 — 사내 봇 운영 시 필수
- [ ] Google Scholar 는 비공식 래퍼만 존재 — **프로덕션 RAG 에는 OpenAlex / Semantic Scholar 권장**
- [ ] 동일 쿼리를 세 도구에 **병렬 호출** 해 서로 다른 관점을 모으는 패턴이 에이전트 라우팅보다 단순할 때도 있다

## 다음

- `08_integration/05_retrievers/04_web_wiki_arxiv.ipynb` — 동일 소스를 **Retriever** 인터페이스로 감싼 쪽. `Document` 반환이 필요할 때.
- `08_integration/07_tools/01_search_web.ipynb` — 실시간 웹 검색과 조합하면 "최신성 + 정의" 커버리지가 완성된다.

## 참고

- NCBI E-utilities: https://www.ncbi.nlm.nih.gov/books/NBK25501/
- arXiv API: https://arxiv.org/help/api
- Wikimedia API etiquette: https://api.wikimedia.org/wiki/Rate_limits